# Un problema de regresión logística

En este laboratorio pondrás a prueba tus conocimientos sobre entrenamiento de modelos de clasificación utilizando regresión logística. Particularmente, realizarás los siguientes procedimientos:

1. Cargar un conjunto de datos.
2. Preparar los datos para el modelado.
3. Realizar la búsqueda de hiperparámetros para la regresión logística.
4. Evaluar los mejores modelos resultantes.

El problema a resolver es el siguiente: dados algunos indicadores de salud de una persona, queremos predecir si está en riesgo de padecer de una enfermedad cardiovascular o no. A lo largo del laboratorio encontrarás múltiples celdas con la siguiente estructura:
```python
#---------- Celda de pruebas - Ejercicio X.X. ----------
# Prueba 1
# Prueba 2
# ...
#-------------------------------------------------------
```
Estas celdas son utilizadas para calificar tus respuestas después de que realices tu envío, por lo que en ellas encontrarás los puntos específicos que serán evaluados. Asegúrate de no incluir código en estas celdas, ya que cualquier modificación será eliminada automáticamente al momento de realizar la evaluación. 

Antes de iniciar, vamos a importar las librerías necesarias:

In [3]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix
from sklearn.pipeline import make_pipeline

from importlib.metadata import version

print(f"Versión de Pandas: {version('pandas')}")
print(f"Versión de Scikit-learn: {version('scikit-learn')}")
print(f"Versión de Numpy: {version('numpy')}")

Versión de Pandas: 2.2.2
Versión de Scikit-learn: 1.7.1
Versión de Numpy: 1.26.4


## 1. Carga de datos

Con las librerías importadas, realizaremos la carga del conjunto de datos:

### Ejercicio 1.1.

Utiliza Pandas para importar el archivo que contiene el conjunto de datos.

* La ruta del archivo .csv es: `./data/heart_disease_health_indicators.csv`, y ya se encuentra en el entorno de Coursera, solo debes importarlo.
* La variable resultante debe tener el nombre `data_raw`.

In [4]:
ruta = 'C:\\Users\\mjgar\\Escritorio\\Maestria\\Semestre_1\\Ciclo_1\\Principios ML\\Semana 5\\Laboratorios\\Taller\\Taller Regresión Logística\\heart_disease_health_indicators.csv'
# your code here
data_raw = pd.read_csv(ruta, sep=',')

## 2. Preparación de datos

Primero vamos a definir la variable `data` para almacenar un conjunto de datos modificado:

In [5]:
data = data_raw.copy()

### Ejercicio 2.1.

Primero vas a realizar la limpieza de los datos. Utiliza Pandas para consultar si existen valores faltantes o filas duplicadas, y elimina estos datos.

* Utiliza funciones sobre el DataFrame para saber si contiene valores faltantes o filas duplicadas. (**Ejemplo: `data.<<Función>>`**)
* Utiliza las funciones de Pandas necesarias para eliminar valores faltantes y filas duplicadas. Asigna tu respuesta a la misma variable `data`. (**Ejemplo: `data = data.<<Función>>`**)
* Encontrarás la línea `data` al final de la celda. Esta línea se usa para que puedas visualizar el resultado, por lo que debes dejarla al final y no debes modificarla.

In [6]:
# your code here
data_falt = data.isna().sum()
data_dupl = data.duplicated().sum()
data = data.dropna().drop_duplicates()
data

,HeartDiseaseorAttack,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,Diabetes,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253675,0.0,1.0,1.0,1.0,45.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,5.0,0.0,1.0,5.0,6.0,7.0
253676,0.0,1.0,1.0,1.0,18.0,0.0,0.0,2.0,0.0,0.0,...,1.0,0.0,4.0,0.0,0.0,1.0,0.0,11.0,2.0,4.0
253677,0.0,0.0,0.0,1.0,28.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,2.0,5.0,2.0
253678,0.0,1.0,0.0,1.0,23.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,0.0,0.0,1.0,7.0,5.0,1.0


In [7]:
#---------- Celda de Pruebas ----------
# La variable 'data' existe
# La variable 'data' es un DataFrame
# La variable 'data' no contiene valores faltantes
# La variable 'data' no contiene filas duplicadas
#--------------------------------------

### Ejercicio 2.2.

Ahora debes dividir el conjunto de datos en entrenamiento y pruebas. Usando el 80% de los datos para entrenar el modelo y el 20% restante para probarlo, utiliza `scikit-learn` para separar el conjunto de datos en dos.

* Guarda tu respuesta en dos variables: `train` y `test`. (**Ejemplo: `train, test = <<Función>>`**)
* Utiliza el parámetro `random_state=0`. Esto hará que la partición sea siempre la misma.
* Encontrarás la línea `train.head()`. Esta línea se usa para que puedas visualizar el resultado del conjunto de entrenamiento. Déjala al final de la celda y no la modifiques.

In [8]:
# your code here
train, test = train_test_split(data, test_size=0.2, random_state=0)
train = train.reset_index(drop=True)
train.head()

,HeartDiseaseorAttack,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,Diabetes,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,0.0,1.0,0.0,24.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,2.0,0.0,0.0,0.0,6.0,6.0,8.0
1,0.0,0.0,0.0,1.0,29.0,1.0,0.0,0.0,1.0,1.0,...,0.0,1.0,3.0,2.0,3.0,0.0,0.0,2.0,5.0,4.0
2,0.0,0.0,0.0,1.0,28.0,0.0,0.0,2.0,0.0,1.0,...,1.0,0.0,3.0,0.0,0.0,1.0,1.0,12.0,4.0,4.0
3,0.0,1.0,0.0,1.0,32.0,1.0,1.0,0.0,0.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,1.0,10.0,4.0,5.0
4,0.0,0.0,1.0,1.0,37.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,3.0,10.0,0.0,0.0,0.0,7.0,6.0,8.0


In [9]:
#---------- Celda de Pruebas ----------
# Las variables "train" y "test" existen
# Las variables "train" y "test" son un DataFrame
# Las variables tienen las dimensiones correctas
#--------------------------------------

### Ejercicio 2.3.

Ahora debes aislar la variable objetivo, `HeartDiseaseorAttack`, de las variables independientes.

* Crea una variable con nombre `x_train` y asígnale la operación necesaria para almacenar solo las variables descriptoras del conjunto de entrenamiento.
* Crea una variable con nombre `y_train` y asígnale la operación necesaria para almacenar la variable objetivo del conjunto de entrenamiento.

In [10]:
# your code here
x_train = train.drop(['HeartDiseaseorAttack'],axis="columns")
y_train = train['HeartDiseaseorAttack']

In [11]:
#---------- Celda de Pruebas ----------
# Las variables "x_train" y "y_train" existen
# La variable "x_train" es un DataFrame
# La variable "y_train" es una Serie de Pandas
# Las variables tienen las dimensiones correctas
#--------------------------------------

### Ejercicio 2.4.

El siguiente paso es realizar la estandarización de nuestros datos. Definiremos un objeto de la clase `StandardScaler()`, que utilizarás para realizar la estandarización, junto con la configuracion .set_output(transform = "pandas") para mantener las columnas nombradas.:

In [12]:
scaler = StandardScaler().set_output(transform="pandas")

Con las variables definidas, realiza la estandarización de los datos. Modifica el conjunto de variables independientes, `x_train`, con las operaciones necesarias para estandarizarlo. Particularmente, deberás seguir estos pasos:

* Utiliza la variable `scaler` para transformar los datos de `x_train`. Asigna el resultado a la misma variable sobreescribiendo su contenido. (**Ejemplo: `x_train = scaler.<<Función>>`**)

In [13]:
# your code here
columns = x_train.columns
x_train = scaler.fit_transform(x_train)
x_train = pd.DataFrame(x_train, columns=columns)

In [14]:
#---------- Celda de Pruebas ----------
# La variable "x_train" es un DataFrame
# La variable tiene las dimensiones correctas
#--------------------------------------

## 3. Búsqueda de hiperparámetros y entrenamiento del modelo

Con el conjunto de datos preparado, es momento de entrenar el modelo de regresión logística. Primero vamos a definir un objeto de la clase `KFold`, con el que definiremos 10 subconjuntos sobre el conjunto de entrenamiento:

In [15]:
kfold = KFold(n_splits=10, shuffle=True, random_state = 0)

También definiremos el objeto de regresión logística:

In [16]:
reglog = LogisticRegression()

### Ejercicio 3.1.

El siguiente paso es definir el espacio de búsqueda de los hiperparámetros.

* Define una variable con el nombre `param_grid` y asígnale la expresión necesaria para crear un diccionario con tres tuplas (**Ejemplo: `param_grid = <<Expresión>>`**):
    * Llave `C` y valor `valores_C`.
    * Llave `penalty` y valor `valores_penalty`.
    * Llave `solver` y valor `valores_solver`.

In [17]:
valores_C = [0.1, 0.5]
valores_penalty = ['l1', 'l2']
valores_solver = ['newton-cg', 'liblinear']
# your code here
param_grid = {'C': valores_C,'penalty':valores_penalty,'solver':valores_solver}

In [18]:
#---------- Celda de Pruebas ----------
# La variable "param_grid" existe
# La variable "param_grid" es un diccionario
# La variable tiene la longitud correcta
#--------------------------------------

### Ejercicio 3.2.

Finalmente, debes crear el objeto de tipo `GridSearchCV` para realizar la búsqueda. Utiliza las variables `reglog`, `param_grid` y `kfold` para definirlo:

* Define una variable con el nombre `grid` y asígnale la función necesaria para crear un objeto de la clase `GridSearchCV`. (**Ejemplo: `grid = <<Función>>`**)
* Utiliza el parámetro `scoring='accuracy'` para que se seleccione el mejor modelo de acuerdo con los valores de exactitud.
* Si quieres, puedes utilizar el parámetro `verbose=3` para que cada paso de la búsqueda se muestre en consola.

In [19]:
# your code here
grid = GridSearchCV(reglog, param_grid, cv=kfold, scoring='accuracy', n_jobs=-1)

In [20]:
#---------- Celda de Pruebas ----------
# La variable "grid" existe
# La variable "grid" es un objeto de la clase GridSearchCV
# La variable "grid" usa la exactitud como método de selección
#--------------------------------------

### Ejercicio 3.3.

A continuación, realiza la búsqueda de hiperparámetros utilizando el conjunto de entrenamiento, compuesto por las variables `x_train` y `y_train`.

* Para este ejercicio no debes asignar tu resultado a ninguna variable. Es decir, solo debes ejecutar una función sobre la variable `grid`, utilizando las variables `x_train` y `y_train` como parámetros. (**Ejemplo: `grid.<<Función>>`**)

In [21]:
# your code here
grid.fit(x_train, y_train)

C:\Users\mjgar\AppData\Roaming\Python\Python312\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
20 fits failed out of a total of 80.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
20 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\mjgar\AppData\Roaming\Python\Python312\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\mjgar\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\mjgar\AppData\Roaming\Python\Python312\site-p

,estimator,LogisticRegression()
,param_grid,"{'C': [0.1, 0.5], 'penalty': ['l1', 'l2'], 'solver': ['newton-cg', 'liblinear']}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,KFold(n_split... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,'l1'


In [22]:
#---------- Celda de Pruebas ----------
# El atributo "best_params_" de la variable "grid" existe
# El atributo "best_estimator_" de la variable "grid" existe
#--------------------------------------

Podemos observar los mejores hiperparámetros usando el atributo `best_params_`:

In [23]:
print("Mejores parámetros: ", grid.best_params_)

Mejores parámetros:  {'C': 0.1, 'penalty': 'l1', 'solver': 'liblinear'}


Además, vamos a definir la variable `mejor_modelo` para realizar las pruebas:

In [24]:
mejor_modelo = grid.best_estimator_

## 4. Evaluación del modelo

Por último, evaluarás el modelo entrenado utilizando el conjunto de pruebas.

### Ejercicio 4.1.

Separa las variables independientes y la variable objetivo en el conjunto de pruebas. Para ello, utiliza Pandas para crear dos variables, `x_test` y `y_test`, que almacenarán las variables independientes y la variable objetivo, respectivamente.

* Crea una variable con nombre `x_test` y asígnale la operación necesaria para almacenar solo las variables independientes del conjunto de pruebas. (**Ejemplo: `x_test = test.<<Función>>`**)
* Crea una variable con nombre `y_test` y asígnale la operación necesaria para almacenar la variable objetivo del conjunto de pruebas. (**Ejemplo: `y_test = <<Consulta>>`**)

In [25]:
# Tu respuesta deben ser dos líneas consecutivas:
#    x_test = test.<<Función>>
#    y_test = <<Consulta>>
# your code here
x_test = test.drop(['HeartDiseaseorAttack'],axis="columns")
y_test = test['HeartDiseaseorAttack']

In [26]:
#---------- Celda de Pruebas ----------
# Las variables existen
# La variable "x_test" es un DataFrame
# La variable "y_test" es una Serie de Pandas
# Las variables tienen las dimensiones correctas
#--------------------------------------

### Ejercicio 4.2.

Realiza la estandarización de los datos del conjunto de pruebas. Modifica el conjunto de variables independientes, `x_test`, con las operaciones necesarias para estandarizarlo. Particularmente, deberás seguir estos pasos:

* Utiliza la variable `scaler` para transformar los datos de `x_test` **utilizando solamente la información del conjunto de entrenamiento**. Asigna el resultado a la misma variable, sobreescribiendo su contenido. (**Ejemplo: `x_test = scaler.<<Función>>`**)

In [27]:
# your code here
columnas = x_test.columns
x_test = scaler.transform(x_test)
x_test = pd.DataFrame(x_test, columns=columns)

In [28]:
#---------- Celda de Pruebas ----------
# La variable "x_test" es un DataFrame
# La variable tiene las dimensiones correctas
#--------------------------------------

### Ejercicio 4.3.

Con el conjunto de pruebas preparado, realiza predicciones con el fin de compararlas con los valores reales almacenados en `y_test`.

* Utiliza la variable `mejor_modelo` para realizar las predicciones sobre el mejor modelo. Asigna el resultado a una variable con nombre `y_pred` (**Ejemplo: `y_pred = mejor_modelo.<<Función>>`**).

In [29]:
# your code here
y_pred = mejor_modelo.predict(x_test)

In [30]:
#---------- Celda de Pruebas ----------
# La variable "y_pred" existe
# La variable "y_pred" es un arreglo
# La variable "y_pred" tiene las dimensiones correctas
#--------------------------------------

### Ejercicio 4.4.

Utiliza el conjunto de predicciones `y_pred` y el conjunto de valores reales `y_test` para obtener la matriz de confusión del mejor modelo.

* Haz un llamado a la función que retorna la matriz de confusión como un arreglo. Asigna el resultado a una nueva variable con el nombre `p44` (**Ejemplo: `p44 = <<Función>>`**).
* Encontrarás la línea `p44` al final de la celda. Esta línea se usa para que puedas visualizar la matriz resultante, por lo que no la debes modificar.

In [31]:
# your code here
p44 = confusion_matrix(y_test, y_pred)
p44

array([[40591,   507],
       [ 4259,   600]], dtype=int64)

In [32]:
#---------- Celda de Pruebas ----------
# La variable "p44" existe
# La variable "p44" es un arreglo de numpy
# La variable "p44" tiene las dimensiones correctas
#--------------------------------------

### Ejercicio 4.5.

Finalmente, utiliza `scikit-learn` para obtener cuatro métricas de rendimiento sobre las predicciones del modelo:

* Define una variable con el nombre `accuracy` y asígnale la función necesaria para obtener la exactitud (`accuracy = <<Función>>`).
* Define una variable con el nombre `recall` y asígnale la función necesaria para obtener la sensibilidad (`recall = <<Función>>`).
* Define una variable con el nombre `precision` y asígnale la función necesaria para obtener la precisión (`precision = <<Función>>`).
* Define una variable con el nombre `f1` y asígnale la función necesaria para obtener el F1-Score (`f1 = <<Función>>`).

In [33]:
# your code here
accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test,y_pred)
precision = precision_score(y_test,y_pred)
f1 = f1_score(y_test,y_pred)
print('-'*10+"Regresión logística"+'-'*10)
print("Exactitud: ", accuracy)
print("Sensibilidad: ", recall)
print('Precisión: ', precision)
print('F1-Score: ', f1)

----------Regresión logística----------
Exactitud:  0.8962943621211132
Sensibilidad:  0.1234821979831241
Precisión:  0.5420054200542005
F1-Score:  0.2011397921555481


In [34]:
#---------- Celda de Pruebas ----------
# Las variables "accuracy", "recall", "precision" y "f1" existen
# Las variables son números
# Las variables tienen valores válidos
#--------------------------------------

## Cierre

Al desarrollar este laboratorio, has reforzado tus capacidades para preparar un conjunto de datos, realizar una búsqueda exhaustiva de hiperparámetros y evaluar el mejor modelo de regresión logística resultante utilizando la matriz de confusión y cuatro métricas de rendimiento. 

---
*Creado por: Nicolás Díaz*

*Última edición: Camilo Rozo*

*Revisado por: Haydemar Nuñez*

*Desarrollado por: María J. García-Bonilla*

*Versión: Febrero 2025*  

*Universidad de los Andes*  